In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.linalg import expm

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part3d_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
ins = pd.read_csv(RAW_DIR / "insured_transitions.csv")

unique_origin = ins.drop_duplicates(subset=["district_id", "origin"])[
    ["district_id", "origin", "exposure_py", "initial_lives"]
]
pooled_exp = unique_origin.groupby("origin")["exposure_py"].sum().to_dict()
pooled_init = unique_origin.groupby("origin")["initial_lives"].sum().to_dict()
pooled_jumps = ins.groupby(["origin", "destination"])["transition_count"].sum().to_dict()

lam_HI = pooled_jumps[("H", "I")] / pooled_exp["H"]
lam_HD = pooled_jumps[("H", "D")] / pooled_exp["H"]
lam_IH = pooled_jumps[("I", "H")] / pooled_exp["I"]
lam_ID = pooled_jumps[("I", "D")] / pooled_exp["I"]

Q = np.array([
    [-(lam_HI + lam_HD),  lam_HI,            lam_HD],
    [ lam_IH,            -(lam_IH + lam_ID),  lam_ID],
    [ 0.0,                0.0,               0.0]
], dtype=float)

P1 = expm(Q * 1.0)
P2 = expm(Q * 2.0)

v0 = np.array([pooled_init.get("H", 0.0), pooled_init.get("I", 0.0), pooled_init.get("D", 0.0)], dtype=float)
occ0 = v0
occ1 = v0 @ P1
occ2 = v0 @ P2

benefit_per_death = 1000.0
deaths_yr1 = occ1[2] - occ0[2]
deaths_yr2 = occ2[2] - occ1[2]

claims_df = pd.DataFrame([
    {
        "period": "Year 1 (t=0 to t=1)",
        "incremental_deaths": deaths_yr1,
        "cumulative_deaths_end": occ1[2],
        "benefit_per_death_usd": benefit_per_death,
        "undiscounted_benefit_usd": deaths_yr1 * benefit_per_death,
    },
    {
        "period": "Year 2 (t=1 to t=2)",
        "incremental_deaths": deaths_yr2,
        "cumulative_deaths_end": occ2[2],
        "benefit_per_death_usd": benefit_per_death,
        "undiscounted_benefit_usd": deaths_yr2 * benefit_per_death,
    },
])

# Show the table first, then save
display(claims_df)
safe_save(lambda p: claims_df.to_csv(p, index=False), "part3d_benefit_payments.csv")

BASE_DIR: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-


,period,incremental_deaths,cumulative_deaths_end,benefit_per_death_usd,undiscounted_benefit_usd
0,Year 1 (t=0 to t=1),616.500631,616.500631,1000.0,616500.631325
1,Year 2 (t=1 to t=2),634.456539,1250.957170,1000.0,634456.539102


Saved: C:\Users\user\Documents\GitHub\Group-4-Assignment-2-Two-\outputs\part3d_benefit_payments.csv
